In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Synthetic Data

We have derived linear regression, but we have not trained it yet. That
takes data, a model, and a training loop in code, and we begin with the
data. Which data should the first run use? Real datasets are what we
ultimately care about, but they make a poor first test. Suppose that we
fit a linear model to house prices like those of
that section and its predictions are poor. Three
explanations compete: the model may be
misspecified, the optimization algorithm or its implementation may be
flawed, or the data may be pathological. On real data we cannot easily tell
which of the three it is.

*Synthetic data* removes the ambiguity by construction. If we generate the
data ourselves, we know the true weights $\mathbf{w}^*$, the true bias
$b^*$, and the distribution of the noise. A model of the right form then has
a known target, and any systematic failure to recover $\mathbf{w}^*$ and
$b^*$, beyond what the noise explains, points to the algorithm or its
implementation. It is good practice, and not only in this chapter, to try a
new learning method first on a problem whose answer we chose ourselves, and
only then on real data.

This section builds such a test for linear regression. We generate examples
from a linear function with the true parameters $\mathbf{w}^* = (2, -3.4)$
and $b^* = 4.2$, and add a little Gaussian noise. We then work out how
close a correct fit can come to these parameters. This is the step that
matters most, since it turns the comparison into a test with a threshold.
The next section trains a linear model on these data and holds its
estimates to the threshold.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Generating the Data

We draw the features of each example independently from a standard normal
distribution, with mean $0$ and variance $1$, and stack the feature vectors
as the rows of the design matrix $\mathbf{X}$. Two features keep the true
parameters easy to compare; nothing below depends on that choice. Each label
is the value of the true linear function plus noise:

$$\mathbf{y} = \mathbf{X}\mathbf{w}^* + b^* + \boldsymbol{\epsilon}.$$

The bias $b^*$ is added to every entry, and the entries of the noise vector
$\boldsymbol{\epsilon}$ are drawn independently from a normal distribution
with mean $0$ and standard deviation $\sigma = 0.01$. Each label thus lies
close to the plane $\mathbf{w}^{*\top}\mathbf{x} + b^*$, typically within a
few hundredths. This is the model of
that section: a linear function with Gaussian noise, for
which minimizing the squared loss is the same as maximizing the likelihood
(that section). A linear model
trained with the squared loss is therefore correctly specified for these
data, and the first of our three explanations is ruled out in advance.

We write the generator as a function, `synthetic_data`, and save it in the
`d2l` library, since later sections use it again. It takes the true
parameters `w` and `b`, the noise level `noise`, and the number `n` of
examples, and returns the design matrix and the labels. The function draws
the features first and the noise second.

In [ ]:
def synthetic_data(w, b, noise, n):
    """n examples with standard normal features and labels Xw + b + noise."""
    X = d2l.randn(n, len(w))
    eps = d2l.randn(n, 1) * noise
    return X, d2l.matmul(X, d2l.reshape(w, (-1, 1))) + b + eps

We draw 2,000 examples with the parameters chosen above: the first 1,000
for training and the other 1,000 for validation. The validation examples
come from the same process but are not used to fit the model. The loss on
them shows how well the model predicts examples it has not seen, a question
we take up in that section. The next section draws
its data in the same way.

In [ ]:
w_star, b_star, noise = d2l.tensor([2, -3.4]), 4.2, 0.01
X, y = synthetic_data(w_star, b_star, noise, 2000)

Each row of `X` holds the two features of one example, and each row of `y`
its label. It is instructive to look at one of them. For the first
example we also compute the noise, the label minus the noise-free value
$\mathbf{w}^{*\top}\mathbf{x} + b^*$.

In [ ]:
x, label = X[0], y[0, 0]
eps = label - (d2l.reduce_sum(w_star * x) + b_star)
print(f'features: ({float(x[0]):.4f}, {float(x[1]):.4f})')
print(f'label: {float(label):.4f}, noise: {float(eps):.4f}')

The label misses the value of the linear function only by the noise, and by
no more than the few hundredths that we expected.

## Recovering the Parameters

A test needs a threshold. Even a perfect implementation does not recover
$\mathbf{w}^*$ and $b^*$ exactly, because the noise moves the labels and,
with them, the minimizer of the loss. How far does it move? The answer
follows from the normal equations of
that section. As there, we append a
column of ones to the $n$ training examples and the bias to the weights:
$\tilde{\mathbf{X}} = [\mathbf{X}, \mathbf{1}]$ has three columns, the
true parameters are $\tilde{\mathbf{w}}^* = (2, -3.4, 4.2)$, and
$\mathbf{y} = \tilde{\mathbf{X}}\tilde{\mathbf{w}}^* + \boldsymbol{\epsilon}$.
The least-squares solution $\hat{\tilde{\mathbf{w}}}$ solves the normal
equations
$\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\hat{\tilde{\mathbf{w}}} = \tilde{\mathbf{X}}^\top\mathbf{y}$.
Substituting the labels gives
$\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}(\hat{\tilde{\mathbf{w}}} - \tilde{\mathbf{w}}^*) = \tilde{\mathbf{X}}^\top\boldsymbol{\epsilon}$,
and therefore

$$\hat{\tilde{\mathbf{w}}} - \tilde{\mathbf{w}}^* = (\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^\top\boldsymbol{\epsilon}.$$

The *estimation error* $\hat{\tilde{\mathbf{w}}} - \tilde{\mathbf{w}}^*$
is a linear function of the noise, so without noise least squares would
recover the parameters exactly.

In general the inverse of $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}$ is
awkward to reason about. For our data, luckily, the matrix is simple. Each
of its entries is a sum over the $n$ examples of a product of two columns,
so $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}/n$ is a matrix of averages.
On the diagonal we average the square of the constant $1$ or of a standard
normal feature, and either has mean $1$. Off the diagonal we average the product of a feature
with the constant or with the other, independent feature, which has mean
$0$. An average over $n = 1{,}000$ examples lies close to its mean, here to
within a few percent. With
$\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} \approx n\mathbf{I}$, the
estimation error becomes an average over the training examples,

$$\hat{\tilde{\mathbf{w}}} - \tilde{\mathbf{w}}^* \approx \frac{1}{n}\tilde{\mathbf{X}}^\top\boldsymbol{\epsilon} = \frac{1}{n}\sum_{i=1}^{n} \epsilon^{(i)}\tilde{\mathbf{x}}^{(i)},$$

where $\tilde{\mathbf{x}}^{(i)} = (\mathbf{x}^{(i)}, 1)$ is the $i$-th row of
$\tilde{\mathbf{X}}$. Each coordinate of the estimation error averages $n$
independent terms $\epsilon^{(i)} \tilde{x}^{(i)}_j$, each with mean $0$
and variance $\sigma^2$, since the noise is independent of the features and
$\tilde{x}_j^2$ has mean $1$.
Such an average has standard deviation $\sigma/\sqrt{n}$, which
that section calls its standard error. With $\sigma = 0.01$ and
$n = 1{,}000$ training examples, every coordinate of
$\hat{\tilde{\mathbf{w}}}$, the bias included, misses its true value by about
$0.01/\sqrt{1000} \approx 3 \times 10^{-4}$.

We check this prediction by solving the least-squares problem on the
training examples with `lstsq`, as in
that section. In the code, `X1` is
$\tilde{\mathbf{X}}$ and `w_hat` is $\hat{\tilde{\mathbf{w}}}$.

In [ ]:
n = 1000                                          # The training examples
X1 = torch.cat((X[:n], torch.ones(n, 1)), 1)      # A column of ones for b
w_hat = torch.linalg.lstsq(X1, y[:n]).solution[:, 0]

We print the estimation errors of the two weights and of the bias, and the
standard error that we predicted for each.

In [ ]:
e_w = d2l.numpy(w_hat[:2] - w_star)
print(f'error in w: ({e_w[0]:.1e}, {e_w[1]:.1e})')
print(f'error in b: {float(w_hat[2] - b_star):.1e}')
print(f'standard error: {noise / n ** 0.5:.1e}')

All three estimation errors are at most a few times $10^{-4}$, the size of
the standard error, as predicted. An error above $10^{-3}$, more than three
standard errors, would be rare. This gives the next section its threshold.
Of course, minibatch stochastic gradient descent does not solve the normal
equations. It approaches the same least-squares estimate, and its random
minibatches keep it fluctuating around that estimate by a similar amount
(the noise ball of that section). A correct
implementation, trained until the loss has converged, should therefore land
within about $10^{-3}$ of the true parameters. An error of $0.1$ cannot be
blamed on the noise: it points to a bug, or to an optimization that has not
converged. The threshold depends on
the data: it shrinks with more examples and less noise (exercise
2), and it assumes
features of unit scale (exercise
3).

Note, however, how little the test claims. Passing it tells us that an
implementation recovers the parameters it should recover, but not how the
method behaves when the model does not match the data, which is the usual
case with real data (exercise
4).

## Summary

A poor result on real data cannot tell a bug from a wrong model; a
synthetic dataset can. For labels generated from a linear function with
known parameters plus Gaussian noise, a linear model with the squared loss
is correctly specified. The noise moves even the exact least-squares
estimate away from the truth, by about the standard error
$\sigma/\sqrt{n}$, here $3 \times 10^{-4}$, so a correct implementation
trained to convergence should land within about $10^{-3}$ of the true
parameters. The test says nothing about data that a linear model does not
fit.


## Exercises

1. [code] **Reproducibility.** Two calls of `synthetic_data` with the same
   arguments return different `X` and `y`. Make them identical
   by fixing the seed with `torch.manual_seed`. State where the call must be
   placed relative to the draws of `X` and `eps`, and explain why seeding
   once at the top of a notebook does not make the second dataset equal to
   the first.

2. [code] **Recovery under noise.** ● Repeat the least-squares fit of
   that section on 1,000 independent
   datasets for each setting below, and measure the standard deviation of
   the estimation error $\hat{w}_1 - w_1^*$ of the first weight.
    1. Vary `noise` over $\{0.001, 0.01, 0.1, 1\}$ with 1,000 training
       examples, and the number $n$ of training examples over
       $\{10, 100, 1{,}000, 10{,}000\}$ with `noise=0.01`. Plot both measurements against the prediction
       $\sigma/\sqrt{n}$ on logarithmic axes.
    1. Find the smallest $n$ at which the measurement agrees with
       $\sigma/\sqrt{n}$ to within 10%, and say which step of the derivation
       fails for fewer examples.
    1. Derive $\textrm{Cov}(\hat{\tilde{\mathbf{w}}}) = \sigma^2(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}})^{-1}$
       from the expression for $\hat{\tilde{\mathbf{w}}} - \tilde{\mathbf{w}}^*$
       in this section. Then generate correlated features
       $x_2 = \rho x_1 + \sqrt{1 - \rho^2}\, z$, with $x_1$ and $z$
       independent and standard normal, for $\rho \in \{0, 0.9, 0.99\}$.
       Predict the standard error of $\hat{w}_1$ from this formula, check
       it numerically, and explain why the predictions
       $\hat{\tilde{\mathbf{w}}}^\top \tilde{\mathbf{x}}$ on new examples drawn from the same
       distribution remain accurate when the weights do not.

3. [code] **A feature on another scale.** Draw the first feature with
   standard deviation 10 instead of 1, and set $w_1^* = 0.2$, so that the
   labels are distributed as before.
    1. Predict $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}/n$, say which
       step of the derivation in this section no longer holds, and predict
       the standard errors of $\hat{w}_1$ and $\hat{w}_2$.
    1. Check both predictions by repeating the least-squares fit on many
       independent datasets.
    1. A bug moves $\hat{w}_1$ by $5 \times 10^{-4}$. Does the threshold
       of $10^{-3}$ catch it? Propose a threshold for each weight that does
       not depend on the scale of its feature.

4. [code] **A model that does not fit.** ● Add the term $c\,x_1^2$ with
   $c = 0.5$ to every label, so that the labels no longer follow a linear
   function of the features, and fit the linear model by least squares as
   in this section.
    1. Using $\mathbb{E}[x_1^2] = 1$ and $\mathbb{E}[x_1^3] = 0$, predict
       which of $\hat{w}_1$, $\hat{w}_2$, and $\hat{b}$ end up away from
       the true values, and by about how much.
    1. Check the prediction for $n \in \{100, 1{,}000, 10{,}000\}$
       training examples. Which estimation errors shrink as $n$ grows, which
       do not, and how do the shrinking ones compare with $\sigma/\sqrt{n}$?
    1. Would the threshold of $10^{-3}$ declare a correct implementation
       faulty on these data? Explain what passing or failing the test of
       this section can tell you when the model does not fit the data.

5. **The loss floor and its fluctuation.** The next section judges a
   training run by its validation loss, the average of
   $\frac{1}{2}(\hat{y} - y)^2$ over the 1,000 validation examples.
    1. Show that a model with the true parameters has expected loss
       $\sigma^2/2$ on a new example, and that no other parameters have a
       smaller one. This is the floor that a correct training run levels
       off at.
    1. For $\epsilon \sim \mathcal{N}(0, \sigma^2)$ one has
       $\mathbb{E}[\epsilon^4] = 3\sigma^4$. Compute the standard deviation
       of the loss $\frac{1}{2}\epsilon^2$ of one example, and from it the
       standard error of the validation loss of the true parameters over
       $n' = 1{,}000$ examples, as a fraction of the floor.
    1. [code] Check both numbers by evaluating the true parameters on many
       independent validation sets. Then use the result of part (a) to find
       the smallest parameter error that raises the expected loss by one
       standard error of the validation loss, and compare it with the
       threshold of $10^{-3}$.

[Discussions](https://d2l.discourse.group/t/6663)